In [16]:
import glob
import pandas as pd
import os
import dask.dataframe as dd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

raw_data_cache = '/Volumes/T7/NEMO-misc'

In [18]:
# More efficient to keep ddf in separate partions
print("Reading filtered Parquet files into a Dask DataFrame...")
ddf = dd.read_parquet(
    '/Volumes/T7/filtered-dispatch-load-data/filtered_dispatch_output')
print("Done reading files.")

# Check available columns to ensure 'SETTLEMENTDATE' exists
print("Columns in the DataFrame:", ddf.columns)
print(ddf.head())

Reading filtered Parquet files into a Dask DataFrame...
Done reading files.
Columns in the DataFrame: Index(['SETTLEMENTDATE', 'DUID', 'INTERVENTION', 'DISPATCHMODE', 'AGCSTATUS',
       'INITIALMW', 'TOTALCLEARED', 'RAMPDOWNRATE', 'RAMPUPRATE', 'LOWER5MIN',
       'LOWER60SEC', 'LOWER6SEC', 'RAISE5MIN', 'RAISE60SEC', 'RAISE6SEC',
       'LOWERREG', 'RAISEREG', 'AVAILABILITY', 'RAISEREGENABLEMENTMAX',
       'RAISEREGENABLEMENTMIN', 'LOWERREGENABLEMENTMAX',
       'LOWERREGENABLEMENTMIN', 'SEMIDISPATCHCAP'],
      dtype='object')
        SETTLEMENTDATE  DUID INTERVENTION DISPATCHMODE AGCSTATUS  INITIALMW  \
0  2009/12/01 00:05:00  BW01            0            0         1   416.3204   
1  2009/12/01 00:05:00  BW02            0            0         1  410.21045   
2  2009/12/01 00:05:00  BW03            0            0         1  417.33316   
3  2009/12/01 00:05:00  BW04            0            0         1  397.92529   
4  2009/12/01 00:05:00  ER01            0            0         1  594

In [21]:
# Work out the most profilic DUIDS to work out their revenue (need to match with price data as estimate)
output_dir = "/Volumes/T7/NEMO-misc"

# 1) Group by DUID and sum RAISEREG and LOWERREG
duid_sums = ddf.groupby('DUID')[['RAISEREG', 'LOWERREG']].sum().compute()

# 2) Create a column for total REG
duid_sums['TOTALREG'] = duid_sums['RAISEREG'] + duid_sums['LOWERREG']

# 5) Find the top 10 DUIDs for each measure
top50_raise = duid_sums.nlargest(50, 'RAISEREG')
top50_lower = duid_sums.nlargest(50, 'LOWERREG')
top50_total = duid_sums.nlargest(50, 'TOTALREG')

print("Top 10 DUID by RAISEREG:")
print(top50_raise)

print("\nTop 10 DUID by LOWERREG:")
print(top50_lower)

print("\nTop 10 DUID by TOTALREG (RAISEREG + LOWERREG):")
print(top50_total)

top50_raise.to_parquet(os.path.join(output_dir, "top10_raise.parquet"))
top50_lower.to_parquet(os.path.join(output_dir, "top10_lower.parquet"))
top50_total.to_parquet(os.path.join(output_dir, "top10_total.parquet"))

Top 10 DUID by RAISEREG:
              RAISEREG      LOWERREG      TOTALREG
DUID                                              
HPRG1     1.531708e+07  8.567010e+04  1.540275e+07
GORDON    1.233578e+07  5.283705e+06  1.761949e+07
BW02      1.123244e+07  8.555857e+06  1.978830e+07
BW01      1.094170e+07  8.708339e+06  1.965004e+07
BW03      1.057701e+07  7.955417e+06  1.853243e+07
BW04      8.928488e+06  6.638276e+06  1.556676e+07
POAT220   8.730493e+06  4.118443e+06  1.284894e+07
GSTONE4   8.695915e+06  4.882825e+06  1.357874e+07
GSTONE3   8.172975e+06  4.652404e+06  1.282538e+07
GSTONE6   7.986112e+06  4.834147e+06  1.282026e+07
GSTONE5   7.657967e+06  4.524401e+06  1.218237e+07
ER04      6.969250e+06  6.200765e+06  1.317001e+07
GSTONE2   6.923058e+06  4.035294e+06  1.095835e+07
ER01      6.887077e+06  5.954798e+06  1.284187e+07
ER02      6.850749e+06  5.693658e+06  1.254441e+07
ER03      6.730666e+06  5.546578e+06  1.227724e+07
GSTONE1   6.582314e+06  3.621828e+06  1.020414e+07
STAN-3

In [23]:
# Create DataFrames
df_raise = pd.read_parquet('/Volumes/T7/NEMO-misc/top10_raise.parquet') 
df_lower = pd.read_parquet('/Volumes/T7/NEMO-misc/top10_lower.parquet') 
df_total = pd.read_parquet('/Volumes/T7/NEMO-misc/top10_total.parquet') 

# Check
print("Top 10 by RAISEREG:\n", df_raise.head(), "\n")
print("Top 10 by LOWERREG:\n", df_lower.head(), "\n")
print("Top 10 by TOTALREG:\n", df_total.head(), "\n")

Top 10 by RAISEREG:
             RAISEREG      LOWERREG      TOTALREG
DUID                                            
HPRG1   1.531708e+07  8.567010e+04  1.540275e+07
GORDON  1.233578e+07  5.283705e+06  1.761949e+07
BW02    1.123244e+07  8.555857e+06  1.978830e+07
BW01    1.094170e+07  8.708339e+06  1.965004e+07
BW03    1.057701e+07  7.955417e+06  1.853243e+07 

Top 10 by LOWERREG:
               RAISEREG      LOWERREG      TOTALREG
DUID                                              
HPRL1     1.035172e+06  1.624365e+07  1.727882e+07
JBUTTERS  4.616561e+06  9.176436e+06  1.379300e+07
BW01      1.094170e+07  8.708339e+06  1.965004e+07
BW02      1.123244e+07  8.555857e+06  1.978830e+07
BW03      1.057701e+07  7.955417e+06  1.853243e+07 

Top 10 by TOTALREG:
             RAISEREG      LOWERREG      TOTALREG
DUID                                            
BW02    1.123244e+07  8.555857e+06  1.978830e+07
BW01    1.094170e+07  8.708339e+06  1.965004e+07
BW03    1.057701e+07  7.955417e+06  1

In [24]:
# Required to join DUIDs to firm names 
generator_info_df = static_table(table_name='Generators and Scheduled Loads', 
                              raw_data_location=raw_data_cache,
                              update_static_file=False)
generator_info_df

INFO: Retrieving static table Generators and Scheduled Loads


,Participant,Station Name,Region,Dispatch Type,Category,Classification,Fuel Source - Primary,Fuel Source - Descriptor,Technology Type - Primary,Technology Type - Descriptor,Aggregation,DUID
0,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,ADPBA1G
1,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Load,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,ADPBA1L
2,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Non-Scheduled,Hydro,Water,Renewable,Run of River,Y,ADPMH1
3,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Semi-Scheduled,Solar,Solar,Renewable,Photovoltaic Tracking Flat panel,Y,ADPPV1
4,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Non-Scheduled,Solar,Solar,Renewable,Photovoltaic Flat panel,Y,ADPPV2
...,...,...,...,...,...,...,...,...,...,...,...,...
527,Tailem Bend II Project Company Pty Ltd as trus...,Tailem Bend 2 Hybrid Renewable Power Station,SA1,Bidirectional Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,TB2B1
528,AGL Macquarie Pty Limited,Broken Hill Battery Energy Storage System,NSW1,Bidirectional Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,BHB1
529,AGL SA Generation Pty Limited,Torrens Island BESS,SA1,Bidirectional Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,TIB1
530,Capital Battery Pty Ltd as Trustee for Capital...,Capital Battery,NSW1,Load,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,CAPBES1


In [25]:
# Suppose df_total is your DataFrame with columns [DUID, RAISEREG, LOWERREG, TOTALREG]
# and generator_info_df is your DataFrame with columns like:
#   [Participant, Station Name, Region, Dispatch Type, Category, Classification, 
#    Fuel Source - Primary, Fuel Source - Descriptor, Technology Type - Primary, 
#    Technology Type - Descriptor, Aggregation, DUID]

# 1) Pick which columns you want from the generator info (optional)
cols_to_keep = [
    "DUID",
    "Participant",
    "Station Name",
    "Region",
    "Dispatch Type",
    "Category",
    "Classification",
    "Fuel Source - Primary",
    "Fuel Source - Descriptor",
    "Technology Type - Primary",
    "Technology Type - Descriptor",
    "Aggregation",
]

# 2) Perform the merge
df_joined = pd.merge(
    df_total, 
    generator_info_df[cols_to_keep],  # or just generator_info_df if you want everything
    on="DUID",
    how="inner"                       # inner join
)

# 3) Inspect the merged DataFrame
print(df_joined.head())

     DUID      RAISEREG      LOWERREG      TOTALREG  \
0    BW02  1.123244e+07  8.555857e+06  1.978830e+07   
1    BW01  1.094170e+07  8.708339e+06  1.965004e+07   
2    BW03  1.057701e+07  7.955417e+06  1.853243e+07   
3  GORDON  1.233578e+07  5.283705e+06  1.761949e+07   
4    BW04  8.928488e+06  6.638276e+06  1.556676e+07   

                  Participant             Station Name Region  \
0   AGL Macquarie Pty Limited  Bayswater Power Station   NSW1   
1   AGL Macquarie Pty Limited  Bayswater Power Station   NSW1   
2   AGL Macquarie Pty Limited  Bayswater Power Station   NSW1   
3  Hydro-Electric Corporation     Gordon Power Station   TAS1   
4   AGL Macquarie Pty Limited  Bayswater Power Station   NSW1   

     Dispatch Type Category Classification Fuel Source - Primary  \
0  Generating Unit   Market      Scheduled                Fossil   
1  Generating Unit   Market      Scheduled                Fossil   
2  Generating Unit   Market      Scheduled                Fossil   
3  Gen

In [30]:
df_joined.shape
pd.set_option('display.max_rows', 100)  # Show up to 100 rows
pd.set_option('display.max_columns', None)  # Show all columns
df_joined.describe()

,RAISEREG,LOWERREG,TOTALREG
count,4.400000e+01,4.400000e+01,4.400000e+01
mean,5.586157e+06,4.235802e+06,9.821959e+06
std,2.676814e+06,2.143307e+06,4.141089e+06
min,1.576824e+06,1.091236e+06,4.180263e+06
25%,3.499780e+06,2.410717e+06,7.085304e+06
50%,5.060943e+06,3.847561e+06,8.216269e+06
75%,6.934606e+06,5.756218e+06,1.282154e+07
max,1.233578e+07,9.176436e+06,1.978830e+07


In [31]:
df_joined

,DUID,RAISEREG,LOWERREG,TOTALREG,Participant,Station Name,Region,Dispatch Type,Category,Classification,Fuel Source - Primary,Fuel Source - Descriptor,Technology Type - Primary,Technology Type - Descriptor,Aggregation
0,BW02,1.123244e+07,8.555857e+06,1.978830e+07,AGL Macquarie Pty Limited,Bayswater Power Station,NSW1,Generating Unit,Market,Scheduled,Fossil,Black Coal,Combustion,Steam Sub-Critical,Y
1,BW01,1.094170e+07,8.708339e+06,1.965004e+07,AGL Macquarie Pty Limited,Bayswater Power Station,NSW1,Generating Unit,Market,Scheduled,Fossil,Black Coal,Combustion,Steam Sub-Critical,Y
2,BW03,1.057701e+07,7.955417e+06,1.853243e+07,AGL Macquarie Pty Limited,Bayswater Power Station,NSW1,Generating Unit,Market,Scheduled,Fossil,Black Coal,Combustion,Steam Sub-Critical,Y
3,GORDON,1.233578e+07,5.283705e+06,1.761949e+07,Hydro-Electric Corporation,Gordon Power Station,TAS1,Generating Unit,Market,Scheduled,Hydro,Water,Renewable,Hydro - Gravity,Y
4,BW04,8.928488e+06,6.638276e+06,1.556676e+07,AGL Macquarie Pty Limited,Bayswater Power Station,NSW1,Generating Unit,Market,Scheduled,Fossil,Black Coal,Combustion,Steam Sub-Critical,Y
5,JBUTTERS,4.616561e+06,9.176436e+06,1.379300e+07,Hydro-Electric Corporation,John Butters Power Station,TAS1,Generating Unit,Market,Scheduled,Hydro,Water,Renewable,Hydro - Gravity,Y
6,GSTONE4,8.695915e+06,4.882825e+06,1.357874e+07,CS Energy Limited,Gladstone Power Station,QLD1,Generating Unit,Market,Scheduled,Fossil,Black Coal,Combustion,Steam Sub-Critical,Y
7,ER04,6.969250e+06,6.200765e+06,1.317001e+07,Origin Energy Electricity Limited,Eraring Power Station,NSW1,Generating Unit,Market,Scheduled,Fossil,Black Coal,Combustion,Steam Sub-Critical,Y
8,POAT220,8.730493e+06,4.118443e+06,1.284894e+07,Hydro-Electric Corporation,Poatina Power Station,TAS1,Generating Unit,Market,Scheduled,Hydro,Water,Renewable,Hydro - Gravity,Y
9,ER01,6.887077e+06,5.954798e+06,1.284187e+07,Origin Energy Electricity Limited,Eraring Power Station,NSW1,Generating Unit,Market,Scheduled,Fossil,Black Coal,Combustion,Steam Sub-Critical,Y


In [ ]:
import pandas as pd

# 1) Group by participant and sum TOTALREG
participant_sums = (
    df_joined
    .groupby("Participant", as_index=False)["TOTALREG"]
    .sum()
)

# 2) Sort in descending order of TOTALREG
participant_sums.sort_values("TOTALREG", ascending=False, inplace=True)

# 3) (Optional) Add a rank column
participant_sums["Rank"] = participant_sums["TOTALREG"].rank(method="dense", ascending=False)

# Show the top 10 participants
print(participant_sums.head(30))

                         Participant      TOTALREG  Rank
3                  CS Energy Limited  8.548674e+07   1.0
1          AGL Macquarie Pty Limited  7.353754e+07   2.0
6         Hydro-Electric Corporation  6.309063e+07   3.0
7  Origin Energy Electricity Limited  5.083354e+07   4.0
9       Stanwell Corporation Limited  5.052586e+07   5.0
0     AGL Loy Yang Marketing Pty Ltd  3.530963e+07   6.0
2      AGL SA Generation Pty Limited  3.049331e+07   7.0
8                Snowy Hydro Limited  1.725507e+07   8.0
4                  Delta Electricity  1.527650e+07   9.0
5            EnergyAustralia Pty Ltd  1.035739e+07  10.0


In [27]:
participant_sums.shape

(10, 3)

In [ ]:
# Define output paths
csv_output_path = os.path.join("/Volumes/T7/NEMO-misc", "participant_sums.csv")
parquet_output_path = os.path.join("/Volumes/T7/NEMO-misc", "participant_sums.parquet")

# Save as CSV
participant_sums.to_csv(csv_output_path, index=False)
print(f"Saved participant_sums to {csv_output_path}")

# (Optional) Save as Parquet
participant_sums.to_parquet(parquet_output_path, index=False)
print(f"Saved participant_sums to {parquet_output_path}")